# ollada — verify diffcoder:8b on a T4 (notebook v0.1.0)

**Before you start:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

This runs ollada's real code (catalog recipe → family adapter → ModelManager) against
Stable-DiffCoder-8B-Instruct in 8-bit. Success = the last cell prints a VERIFICATION RECORD.
If anything crashes, the failing cell *is* the bug report — leave it and tell Muse.
Generation uses the recipe's provisional defaults (steps=128; the vendor example uses 512). Watch item: the model's block-mask code builds its mask in bfloat16 while the recipe loads the model in float16 on a T4 — if a dtype error appears, that *is* a finding; leave the cell and report it.


In [ ]:
# 1 — install (DiffCoder family: transformers 5.3.0 (the version its config was saved with))
import subprocess, sys
pkgs = [
    "transformers==5.3.0", "torch", "accelerate", "bitsandbytes", "pyyaml",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "git+https://github.com/OpenSourceRulesz/ollada@main"], check=True)
print("installed")

In [ ]:
# 2 — version stamp + GPU check (acceptance gate)
import torch, ollada
NOTEBOOK_VERSION = "0.1.0"  # also used by the record cell — keep it a variable, not a literal (dream v0.1.0 bug)
print("notebook version:", NOTEBOOK_VERSION)
print("ollada version:", ollada.__version__)
assert torch.cuda.is_available(), "no CUDA — set the runtime to T4 GPU first"
name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  VRAM: {total:.1f} GB")

In [ ]:
# 3 — load diffcoder:8b through ollada itself (timed)
import time, torch
from ollada.runtime.manager import ModelManager

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
manager = ModelManager()
adapter = manager.ensure_loaded("diffcoder:8b")
load_s = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"loaded diffcoder:8b in {load_s:.1f}s — peak VRAM {peak:.2f} GB")
print("ready on cuda")

In [ ]:
# 4 — fixed prompt set (coding set: generation / bug-fix / completion / one general sanity check)
prompts = [
    "Write a Python function that returns the nth Fibonacci number, with a docstring.",
    "This function is supposed to return the average of a list of numbers, but it crashes on an empty list and has another bug. Find and fix both bugs, and show the corrected function:\n\ndef average(nums):\n    total = 0\n    for n in nums:\n        total =+ n\n    return total / len(nums)",
    "Complete this Python function so it checks whether a string is a palindrome, ignoring case and non-alphanumeric characters. Only output the completed code:\n\ndef is_palindrome(s):\n    # your code here",
    "In one or two sentences, what does a hash map do?",
]
results = []
for p in prompts:
    r = manager.chat("diffcoder:8b", [{"role": "user", "content": p}])
    tps = r.tokens / r.seconds if r.seconds else 0
    results.append({"prompt": p[:40], "tokens": r.tokens, "seconds": round(r.seconds, 2), "tok_per_s": round(tps, 1)})
    print("=" * 70)
    print("PROMPT:", p)
    print(f"[{r.tokens} tokens in {r.seconds:.1f}s = {tps:.1f} tok/s]")
    print(r.text)

In [ ]:
# 5 — VERIFICATION RECORD (paste this whole block back to Muse)
import json, torch
record = {
    "model": "diffcoder:8b",
    "notebook_version": NOTEBOOK_VERSION,
    "ollada_version": ollada.__version__,
    "hardware": torch.cuda.get_device_name(0),
    "load_seconds": round(load_s, 1),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "runs": results,
}
print(json.dumps(record, indent=2))